In [5]:
import json
from transformers import AutoTokenizer
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig
import numpy as np

def compile_predictions(predictions_file, model_name):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    results = []
    total_tokens = 0
    with open(predictions_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            gold = parse(f"${data['answer']}$", extraction_config=extraction_target)
            
            # Get model generation output (usually first item)
            llm_output = data['model_generation'][0] if isinstance(data['model_generation'], list) else data['model_generation']
            answer = parse(llm_output, extraction_config=extraction_target)
            total_tokens += len(tokenizer.encode(llm_output))
            result = verify(gold, answer)
            results.append(result)
    
    accuracy = sum(results) / len(results) if results else 0
    avg_tokens = total_tokens / len(results) if results else 0
    return accuracy, avg_tokens


In [6]:
# limit of RLVR, unbiased estimation
def compile_predictions_passK(predictions_file, model_name, total_questions_num, n, seed_max):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    
    # jsonl file, each line is a json object
    # each json object has a key 'model_generation', the number of generations is N
    N = n * seed_max
    
    def read_scores_from_jsonl(file_path):
        try:
            with open(file_path, 'r') as f:
                data = [json.loads(line) for line in f]
            
            scores = []
            for item in data:
                gold = parse(f"${item['answer']}$", extraction_config=extraction_target)
                item_scores = []
                for generation in item['model_generation']:
                    answer = parse(generation, extraction_config=extraction_target)
                    result = verify(gold, answer)
                    item_scores.append(result)
                scores.append(item_scores)
            return scores
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
        return []
    
    def read_jsonl_file(file_path, model_name, True_n_ls, total_questions_num, N, seed_max):
        for seed in range(1, seed_max + 1):
            scores = read_scores_from_jsonl(file_path)
            for i in range(total_questions_num):
                for j in range(n):
                    True_n_ls[(seed-1)*n+j][i] = int(scores[i][j])
            
    def pass_at_k(n, c, k):
        if n - c < k: return 1.0
        return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

    True_n_ls = [[0 for i in range(total_questions_num)] for j in range(N)]
    read_jsonl_file(predictions_file, model_name, True_n_ls, total_questions_num, N, seed_max)
    True_n_ls = np.array(True_n_ls)
    
    correct_ls = [0 for i in range(total_questions_num)]
    for i in range(total_questions_num):
        for j in range(n * seed_max):
            correct_ls[i] += True_n_ls[j][i]    
    # print("correct_ls: ", correct_ls)
    
    pass_at_k_ls = [[] for i in range(N)]
    for i in range(N):
        for j in range(total_questions_num):
            pass_at_k_ls[i].append(pass_at_k(n * seed_max, correct_ls[j], i+1))
    # print("pass_at_k_ls: ", pass_at_k_ls)
    
    pass_at_k_mean_ls = [np.mean(pass_at_k_ls[i]) for i in range(N)]
    
    pass_at_k_mean_ls = np.array(pass_at_k_mean_ls)
            
    return pass_at_k_mean_ls, True_n_ls
    
    '''
    pass_at_k_mean_ls is a numpy array, pass_at_k_mean_ls[k-1] means pass@k 
    '''

In [7]:
import matplotlib.pyplot as plt

def draw_comparison_fig(baseline_x, baseline_y, seal_x, seal_y, x_label, y_label, title):
    plt.figure(figsize=(10, 5))
    plt.plot(baseline_x, baseline_y, marker='o', linestyle='-', color='b', label='Baseline')
    plt.plot(seal_x, seal_y, marker='o', linestyle='-', color='r', label='SEAL')
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.ylim(bottom=0, top=1)
    plt.legend()
    plt.show()

## Example

In [8]:
import os
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
paths = ["/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-3/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime24/100/0.1/10000/1/predictions.jsonl",
         "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-6/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime24/100/0.1/10000/1/predictions.jsonl",
         "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-27/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime24/100/0.1/10000/1/predictions.jsonl",
         "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-57/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime24/100/0.1/10000/1/predictions.jsonl",
         "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-3/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime25/100/0.1/10000/1/predictions.jsonl",
         "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-6/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime25/100/0.1/10000/1/predictions.jsonl",
         "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-27/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime25/100/0.1/10000/1/predictions.jsonl",
         "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-basic/reason-gguf-57/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/aime25/100/0.1/10000/1/predictions.jsonl"]
for path in paths:
    metric_path = os.path.join(os.path.dirname(path), "metrics.json")
    if os.path.exists(metric_path):
        continue
    accuracy, avg_tokens = compile_predictions(path, model_name)
    
    # number of questions, # of lines in the jsonl file
    with open(path, 'r') as f:
        total_questions_num = sum(1 for line in f)
    # number of generations
    with open(path, 'r') as f:
        for line in f:
            data = json.loads(line)
            n = len(data['model_generation'])
            break
    seed_max = 1 # number of seeds
    passK, True_n_ls = compile_predictions_passK(path, model_name, total_questions_num, n, seed_max)
    # pass@K, all the pass@K values
    passK_dict = {}
    for i in range(len(passK)):
        # four decimal places, show with value without np.float64
        passK_dict[f"pass{i+1}"] = f"{passK[i]:.4f}"
    
    json_output = {
        "accuracy": accuracy,
        "avg_tokens": avg_tokens,
        "passK": passK_dict,
        "true_n_ls": True_n_ls.tolist()
    }

    with open(metric_path, 'w') as f:
        f.write(json.dumps(json_output))


In [9]:
import os
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
paths = []
for path in paths:
    metric_path = os.path.join(os.path.dirname(path), "metrics.json")
    if os.path.exists(metric_path):
        continue
    accuracy, avg_tokens = compile_predictions(path, model_name)
    
    # number of questions, # of lines in the jsonl file
    with open(path, 'r') as f:
        total_questions_num = sum(1 for line in f)
    # number of generations
    with open(path, 'r') as f:
        for line in f:
            data = json.loads(line)
            n = len(data['model_generation'])
            break
    seed_max = 1 # number of seeds
    passK, True_n_ls = compile_predictions_passK(path, model_name, total_questions_num, n, seed_max)
    # pass@K, all the pass@K values
    passK_dict = {}
    for i in range(len(passK)):
        # four decimal places, show with value without np.float64
        passK_dict[f"pass{i+1}"] = f"{passK[i]:.4f}"
    
    json_output = {
        "accuracy": accuracy,
        "avg_tokens": avg_tokens,
        "passK": passK_dict,
        "true_n_ls": True_n_ls
    }

    with open(metric_path, 'w') as f:
        f.write(json.dumps(json_output))
